# THORS - 2D Virtual Try-On Server (FASHN VTON v1.5)

Notebook này biến **Google Colab** hoặc **Kaggle** thành một **API Server miễn phí** cho tính năng thử đồ 2D của dự án THORS.

### Hướng dẫn chạy nhanh (3 bước)
1. **Bật GPU:**
   - **Google Colab:** Menu `Runtime` → `Change runtime type` → chọn **T4 GPU** → `Save`.
   - **Kaggle:** Panel bên phải `Session options` → `Accelerator` chọn **GPU T4 x2** (hoặc **GPU P100**), và bật **Internet: On** (cần xác minh SĐT tài khoản Kaggle trước).
2. Bấm **Run All** (Chạy tất cả các cell từ trên xuống dưới).
3. Đợi cell cuối cùng in ra link `https://xxxx.trycloudflare.com`:
   - Mở `https://xxxx.trycloudflare.com/docs` để test upload 2 ảnh trực tiếp trên trình duyệt.
   - Hoặc copy link `https://xxxx.trycloudflare.com` dán vào cấu hình backend Spring Boot để gọi `POST /try-on`.

In [ ]:
# @title 1. Kiểm tra GPU & Cài đặt môi trường
import os
import subprocess
import sys
from pathlib import Path

gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError(
        "Không tìm thấy GPU! Hãy bật GPU T4 trong Settings của Colab/Kaggle rồi chạy lại."
    )
print(gpu_check.stdout.splitlines()[0:10])

WORK_DIR = Path.cwd() / "fashn-vton-1.5"
if not WORK_DIR.exists():
    print("Đang clone fashn-AI/fashn-vton-1.5...")
    subprocess.run(
        ["git", "clone", "https://github.com/fashn-AI/fashn-vton-1.5.git", str(WORK_DIR)],
        check=True,
    )

os.chdir(WORK_DIR)
print("Thư mục làm việc:", Path.cwd())

# Cài đặt thư viện fashn-vton và web server FastAPI
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "python-multipart"],
    check=True,
)

# Tải cloudflared (tạo HTTPS public URL miễn phí, không cần tài khoản)
cloudflared_bin = Path("/usr/local/bin/cloudflared")
if not cloudflared_bin.exists():
    print("Đang tải cloudflared...")
    subprocess.run(
        [
            "curl",
            "-L",
            "-sS",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-o",
            str(cloudflared_bin),
        ],
        check=True,
    )
    cloudflared_bin.chmod(0o755)

print("Cài đặt xong!")

In [ ]:
# @title 2. Tải weights (~2 GB) & Khởi tạo FASHN VTON v1.5 Pipeline
import subprocess
import sys
from pathlib import Path
from fashn_vton import TryOnPipeline

WEIGHTS_DIR = Path.cwd() / "weights"
if not (WEIGHTS_DIR / "model.safetensors").exists():
    print("Đang tải model weights (~2 GB), vui lòng đợi 1-2 phút...")
    subprocess.run(
        [sys.executable, "scripts/download_weights.py", "--weights-dir", str(WEIGHTS_DIR)],
        check=True,
    )

print("Đang nạp model lên GPU...")
pipeline = TryOnPipeline(weights_dir=str(WEIGHTS_DIR))
print("Pipeline sẵn sàng!")

In [ ]:
# @title 3. (Tùy chọn) Test thử nhanh bằng ảnh mẫu trong repo
from pathlib import Path
from PIL import Image
from IPython.display import display

data_dir = Path.cwd() / "examples" / "data"
person_candidates = sorted(data_dir.glob("model.*")) if data_dir.exists() else []
garment_candidates = sorted(data_dir.glob("garment.*")) if data_dir.exists() else []

if person_candidates and garment_candidates:
    person_img = Image.open(person_candidates[0]).convert("RGB")
    garment_img = Image.open(garment_candidates[0]).convert("RGB")

    out = pipeline(
        person_image=person_img,
        garment_image=garment_img,
        category="tops",
        garment_photo_type="model",
        num_timesteps=25,
        guidance_scale=1.5,
        seed=42,
    )
    result_img = out.images[0] if hasattr(out, "images") else out
    print("Ảnh mẫu đầu vào -> Kết quả thử đồ:")
    display(result_img)
else:
    print("Không thấy ảnh mẫu trong examples/data, bỏ qua bước test nhanh.")

In [ ]:
# @title 4. Chạy API Server (/try-on) & Mở link Cloudflare Tunnel
import asyncio
import io
import re
import subprocess
import threading
import time
from typing import Literal

import torch
import uvicorn
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from PIL import Image

app = FastAPI(
    title="THORS Virtual Try-On API (FASHN VTON v1.5)",
    version="1.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

gpu_lock = threading.Lock()


@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": "fashn-vton-1.5",
        "cuda_available": torch.cuda.is_available(),
    }


def _run_try_on(
    person_img: Image.Image,
    garment_img: Image.Image,
    category: str,
    garment_photo_type: str,
    num_timesteps: int,
    guidance_scale: float,
    seed: int,
) -> bytes:
    with gpu_lock:
        out = pipeline(
            person_image=person_img,
            garment_image=garment_img,
            category=category,
            garment_photo_type=garment_photo_type,
            num_timesteps=num_timesteps,
            guidance_scale=guidance_scale,
            seed=seed,
        )
    result_img = out.images[0] if hasattr(out, "images") else out
    buf = io.BytesIO()
    result_img.save(buf, format="PNG")
    return buf.getvalue()


@app.post("/try-on")
async def try_on(
    person_image: UploadFile = File(..., description="Ảnh người cần thử đồ"),
    garment_image: UploadFile = File(..., description="Ảnh quần/áo sản phẩm"),
    category: Literal["tops", "bottoms", "one-pieces"] = Form("tops"),
    garment_photo_type: Literal["flat-lay", "model"] = Form("flat-lay"),
    num_timesteps: int = Form(30, ge=10, le=50),
    guidance_scale: float = Form(1.5, ge=1.0, le=5.0),
    seed: int = Form(42),
):
    try:
        person_bytes = await person_image.read()
        garment_bytes = await garment_image.read()
        person_pil = Image.open(io.BytesIO(person_bytes)).convert("RGB")
        garment_pil = Image.open(io.BytesIO(garment_bytes)).convert("RGB")
    except Exception as exc:
        raise HTTPException(status_code=400, detail=f"Ảnh đầu vào không hợp lệ: {exc}")

    start = time.perf_counter()
    loop = asyncio.get_running_loop()
    png_bytes = await loop.run_in_executor(
        None,
        _run_try_on,
        person_pil,
        garment_pil,
        category,
        garment_photo_type,
        num_timesteps,
        guidance_scale,
        seed,
    )
    elapsed = time.perf_counter() - start

    return StreamingResponse(
        io.BytesIO(png_bytes),
        media_type="image/png",
        headers={"X-Inference-Seconds": f"{elapsed:.2f}"},
    )


PORT = 8000
config = uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning")
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

time.sleep(2)
print(f"FastAPI đang chạy tại http://127.0.0.1:{PORT}. Đang mở Cloudflare Tunnel...")

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

url_printed = False
try:
    assert tunnel_proc.stdout is not None
    for line in tunnel_proc.stdout:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match and not url_printed:
            public_url = match.group(0)
            url_printed = True
            print("=" * 60)
            print(f"PUBLIC API URL : {public_url}")
            print(f"TEST TRÊN WEB  : {public_url}/docs")
            print(f"HEALTH CHECK   : {public_url}/health")
            print(f"ENDPOINT TRYON : POST {public_url}/try-on")
            print("=" * 60)
            print("Giữ cell này chạy trong suốt lúc test hoặc demo. Bấm Stop để tắt.")
except KeyboardInterrupt:
    print("Đang dừng server...")
finally:
    tunnel_proc.terminate()
    server.should_exit = True
